# Lab 4, schema enforcement and evolution

Delta enforces schema by default, meaning a write with a mismatched schema just gets rejected instead of quietly going through. I want to actually prove that happens, then show the two proper ways around it, mergeSchema for adding or widening columns, and column mapping for renaming or dropping them safely. At the end I'll talk a bit about data contracts as the more controlled alternative to just leaving evolution wide open.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "lab4", "Catalog")
catalog = dbutils.widgets.get("catalog")

## Proving enforcement actually blocks a bad write

I'm taking the real sales order lines table and trying to append a dataframe with an extra column that doesn't exist in the target, with no mergeSchema option set. This should fail, and that failure is the whole point of this cell, not something to fix.

In [0]:
test_extra_col_df = (
    spark.table(f"{catalog}.silver.slv_sales_order_lines")
    .limit(5)
    .withColumn("customer_segment_note", F.lit("test"))
)

try:
    (test_extra_col_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.silver.slv_sales_order_lines")
    )
    print("Write succeeded, that's unexpected, enforcement should have blocked this")
except Exception as e:
    print("Write rejected, as expected:")
    print(str(e)[:500])

## Now the controlled way past it

Same write, same extra column, but this time with mergeSchema explicitly turned on. The difference matters, enforcement is the safe default and mergeSchema is something I have to opt into on purpose, it's not something that just happens by accident.

In [0]:
(test_extra_col_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{catalog}.silver.slv_sales_order_lines")
)

print("Write succeeded with mergeSchema=true")
spark.sql(f"DESCRIBE TABLE {catalog}.silver.slv_sales_order_lines").show(truncate=False)

## Checking the old rows are fine

This is really the mechanism that makes schema evolution safe, old rows don't get rewritten or broken, they just don't have a value for a column that didn't exist yet when they were written.

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customer_segment_note IS NULL THEN 1 ELSE 0 END) AS null_new_column,
    SUM(CASE WHEN customer_segment_note IS NOT NULL THEN 1 ELSE 0 END) AS populated_new_column
FROM lab4.silver.slv_sales_order_lines

## Widening a column type

In [0]:
price_widened_df = (
    spark.table(f"{catalog}.silver.slv_sales_order_lines")
    .limit(5)
    .withColumn("unit_price", F.col("unit_price").cast("double"))
)

(price_widened_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{catalog}.silver.slv_sales_order_lines")
)

print("Widening write succeeded")
spark.sql(f"DESCRIBE TABLE {catalog}.silver.slv_sales_order_lines").filter("col_name = 'unit_price'").show(truncate=False)

## Column mapping, renaming and dropping safely

mergeSchema only ever adds columns. Renaming or dropping is riskier, without column mapping turned on Delta has to physically rewrite every data file just to remove or rename one column. Column mapping decouples the logical column name from how it's actually stored, so rename and drop become quick metadata only changes instead of a full rewrite.


In [0]:
%sql
ALTER TABLE lab4.silver.slv_sales_order_lines
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5'
)

## Checking incoming data against an explicit expected schema before writing, and failing loudly if it doesn't match

This pairs with something added upstream in `02_silver_customers` and `03_silver_sales_orders`: rows that fail a required cast now go to a quarantine table instead of silently becoming null or blocking the whole load. A contract check like the one below and a quarantine table are solving adjacent but different problems, so the contract catches *structural* drift (a column that shouldn't be there, or one that's missing), while quarantine catches *row-level* data quality problems within an otherwise-expected schema.

In [0]:
expected_schema = {
    "order_number", "product_id", "customer_id", "customer_name", "order_datetime",
    "number_of_line_items", "product_name", "unit_price", "quantity", "unit",
    "currency", "promo_id", "promo_discount", "line_revenue", "silver_updated_at",
}


def check_contract(df, expected_columns, table_name):
    actual_columns = set(df.columns)
    unexpected = actual_columns - expected_columns
    missing = expected_columns - actual_columns

    if unexpected or missing:
        raise ValueError(
            f"Contract violation for {table_name}: "
            f"unexpected columns={unexpected or 'none'}, missing columns={missing or 'none'}. "
            f"Update the contract on purpose before allowing this write."
        )
    print(f"Contract check passed for {table_name}")


# should pass, matches the real current schema
check_contract(spark.table(f"{catalog}.silver.slv_sales_order_lines"), expected_schema, "slv_sales_order_lines")

# should fail, checking the mismatched test dataframe on purpose to prove it actually catches drift
try:
    check_contract(test_extra_col_df, expected_schema, "slv_sales_order_lines (with extra column)")
except ValueError as e:
    print(f"Caught as expected: {e}")